# 🤖 35 · AI Agents & Multimodal Integration for Bioimage Analysis
### Orchestrating a toolbox of models and fusing imaging with other data, responsibly

**Part J · Research frontier** · Module 35 of 37 (frontier) · builds on 28, 30, 31, 34

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/35_ai_agents_and_multimodal_pipelines.ipynb)

| | |
|---|---|
| **Hardware target** | CPU (every demo finishes in well under a minute; no network access or live LLM API calls are used anywhere in this notebook) |
| **Core frameworks** | NumPy, Matplotlib, PyTorch |
| **Prerequisites** | Modules 28, 30, 31 (conceptually — referenced, not re-taught) and Module 34 (trustworthy AI) |
| **Builds on** | Modules 28, 30, 31, 34 |
| **Research frontier covered** | Model zoos, agentic tool orchestration, retrieval-grounded assistants, multimodal (imaging + omics/clinical) fusion |
| **Estimated time** | 75–95 min |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> **All data are SIMULATED.** The "agent" in this notebook is a small, fully deterministic, scripted program — not a live language model — so every step of its plan-act-observe loop, and every number it reports, can be checked exactly against known ground truth.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain what a model zoo (BioImage.IO-style) is, and why standardized model metadata becomes
  essential once many interchangeable models (Module 28) exist.
- Trace a concrete plan → call-tool → observe-result → next-step ("ReAct"-style) loop that chains
  this course's own toolkit functions to answer a bioimage-analysis request, and explain why each
  step needs its output checked before the next step trusts it.
- Build a tiny retrieval step that grounds an answer in real documentation text, and show it beats
  guessing without retrieval on a fixed, checkable set of queries.
- Demonstrate, with a controlled synthetic example, a case where neither an image-derived feature
  nor an omics-derived feature alone predicts an outcome, but a model that reasons over both
  jointly can.
- List the responsible-use practices (output verification, dual-use and privacy awareness) that
  matter more, not less, once a pipeline — not a single model — is making decisions.

## 🗺️ Contents

1. Model zoos: a shared registry of interchangeable tools
2. An agent loop: plan, act, observe, verify
3. Retrieval-augmented analysis assistants
4. Fusing imaging with multi-omics and clinical metadata
5. Responsible use: verification, dual-use and privacy
6. 🧭 From toy to real data

## 🧫 The biological question

Modules 28–31 each gave you one specialized capability: a foundation model that segments anything
(28), a network that predicts a stain it never saw (29), a pipeline that profiles a cell's whole
morphology (30), a way to reason about cells in their spatial neighborhood (31). A real research
question rarely needs just one of these — "does this drug change which cells cluster together,
and does that match their gene expression?" needs segmentation, profiling, spatial reasoning and a
second data modality, chained together and cross-checked. This module is about *orchestrating*
that toolbox — ideally with an AI agent doing the chaining — without losing the habit, from
Module 34, of checking every step's output before trusting the next one.

---

## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
try:
    import torch
    print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not installed yet — the next cell installs it if needed.')


In [ ]:
!pip install -q scikit-image torch
# ── Dependencies ── (Colab already has most of these; we install only what is missing)


In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
SEED = 27
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11


### 🧰 Course toolkit

The exact same helper functions used throughout this course, copied verbatim so this notebook runs on its own.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

SHAPE_CLASSES = ("round", "elongated", "kidney", "ring", "dividing", "irregular")


def _draw_blob(canvas: np.ndarray, cy: float, cx: float, r: float, shape: str,
                rng: np.random.Generator, angle: float = 0.0) -> None:
    """Add one synthetic nucleus/cell of the given shape class onto ``canvas`` (in place, additive)."""
    size = canvas.shape[0]
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    y, x = yy - cy, xx - cx
    ca, sa = np.cos(angle), np.sin(angle)
    yr, xr = ca * y + sa * x, -sa * y + ca * x  # rotate into the blob's own frame
    if shape == "round":
        d = (yr / r) ** 2 + (xr / r) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "elongated":
        d = (yr / (r * 0.55)) ** 2 + (xr / (r * 1.7)) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "kidney":
        d = (yr / r) ** 2 + (xr / r) ** 2
        dent = np.exp(-((yr - 0.0) ** 2 + (xr - r * 0.9) ** 2) / (2 * (r * 0.5) ** 2))
        canvas += np.clip(np.exp(-d * 2.0) - 0.9 * dent, 0, None)
    elif shape == "ring":
        d = np.sqrt(yr ** 2 + xr ** 2)
        canvas += np.exp(-((d - r) ** 2) / (2 * (r * 0.28) ** 2))
    elif shape == "dividing":
        sep = r * 1.1
        for sign in (-1, 1):
            d = ((yr - sign * sep * 0.5) / (r * 0.75)) ** 2 + (xr / (r * 0.75)) ** 2
            canvas += np.exp(-d * 2.0)
    else:  # irregular: a round blob perturbed by a few random bumps
        d = (yr / r) ** 2 + (xr / r) ** 2
        base = np.exp(-d * 2.0)
        for _ in range(3):
            bx, by = rng.uniform(-r, r, size=2)
            bump = np.exp(-(((yr - by) ** 2 + (xr - bx) ** 2) / (2 * (r * 0.45) ** 2)))
            base = np.clip(base + 0.5 * bump, 0, 1.3)
        canvas += base


def add_gaussian_noise(image: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Additive Gaussian (read) noise: the classic noise floor of a camera sensor."""
    return image + rng.normal(0.0, sigma, size=image.shape)


def blur_image(image: np.ndarray, psf: np.ndarray) -> np.ndarray:
    """Convolve an image with a point-spread function (microscope optical blur)."""
    return ndi.convolve(image, psf, mode="reflect")


def gaussian_psf(sigma: float, radius: int | None = None) -> np.ndarray:
    """A normalised 2-D Gaussian point-spread function (the microscope's blur kernel)."""
    if radius is None:
        radius = max(1, int(np.ceil(3 * sigma)))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return k / k.sum()


def simulate_nuclei_image(n_cells: int = 25, size: int = 160, touching: bool = False,
                           shape_classes: tuple[str, ...] | None = None,
                           psf_sigma: float = 1.2, background: float = 0.05,
                           noise_sigma: float = 0.03, seed: int = 0):
    """A SIMULATED single-channel fluorescence image of cell nuclei/cells.

    Returns ``image`` (size x size, background + PSF-blurred blobs + Gaussian noise),
    ``instance_mask`` (0 = background, 1..n_cells = one integer id per cell, the ground truth
    for segmentation/detection/tracking), ``centroids`` (n_cells x 2, row, col), ``shape_labels``
    (one of SHAPE_CLASSES per cell, the ground truth for classification), and ``boxes`` (n_cells x 4,
    ymin, xmin, ymax, xmax — the ground truth for detection).
    """
    rng = np.random.default_rng(seed)
    classes = shape_classes or SHAPE_CLASSES
    image = np.full((size, size), background, dtype=float)
    instance_mask = np.zeros((size, size), dtype=int)
    centroids, labels, boxes = [], [], []
    margin = size * 0.12
    min_sep = size * 0.11 if not touching else size * 0.065
    placed = []
    attempts = 0
    while len(placed) < n_cells and attempts < n_cells * 60:
        attempts += 1
        cy, cx = rng.uniform(margin, size - margin, size=2)
        if all(np.hypot(cy - py, cx - px) > min_sep for py, px in placed):
            placed.append((cy, cx))
    for i, (cy, cx) in enumerate(placed):
        shape = classes[rng.integers(len(classes))]
        r = rng.uniform(size * 0.028, size * 0.045)
        angle = rng.uniform(0, 2 * np.pi)
        single = np.zeros_like(image)
        _draw_blob(single, cy, cx, r, shape, rng, angle)
        image += single
        this_mask = single > 0.25
        instance_mask[this_mask & (instance_mask == 0)] = i + 1
        centroids.append((cy, cx))
        labels.append(shape)
        ys, xs = np.where(this_mask)
        if len(ys):
            boxes.append((ys.min(), xs.min(), ys.max(), xs.max()))
        else:
            boxes.append((cy - r, cx - r, cy + r, cx + r))
    image = blur_image(image, gaussian_psf(psf_sigma))
    image = add_gaussian_noise(image, noise_sigma, rng)
    image = np.clip(image, 0, None)
    return dict(image=image, instance_mask=instance_mask, centroids=np.array(centroids),
                shape_labels=np.array(labels), boxes=np.array(boxes), source="SIMULATED")


def watershed_labels(image: np.ndarray, min_distance: int = 7) -> np.ndarray:
    """Marker-controlled watershed segmentation of a blobby intensity image: finds markers at
    local intensity maxima (one per expected object) and floods from them. Returns an integer
    label image (0 = background)."""
    from skimage.feature import peak_local_max
    from skimage.segmentation import watershed as _watershed
    smooth = ndi.gaussian_filter(image, 1.0)
    coords = peak_local_max(smooth, min_distance=min_distance, labels=smooth > smooth.mean())
    markers = np.zeros(image.shape, dtype=int)
    for i, (y, x) in enumerate(coords, start=1):
        markers[y, x] = i
    markers = ndi.label(ndi.binary_dilation(markers > 0, iterations=2))[0] if coords.size else markers
    elevation = -smooth
    return _watershed(elevation, markers=markers, mask=smooth > smooth.mean() * 0.6)


def shape_features(mask: np.ndarray) -> dict:
    """Simple morphological/shape features of a binary blob mask: area, perimeter (boundary
    pixel count), equivalent-circle compactness, and the aspect ratio of its bounding box."""
    area = mask.sum()
    eroded = ndi.binary_erosion(mask)
    perimeter = max(1, (mask & ~eroded).sum())
    compactness = 4 * np.pi * area / (perimeter ** 2 + 1e-9)  # 1.0 for a perfect circle
    ys, xs = np.nonzero(mask)
    if len(ys) == 0:
        return dict(area=0, perimeter=0, compactness=0.0, aspect_ratio=1.0)
    h = ys.max() - ys.min() + 1
    w = xs.max() - xs.min() + 1
    aspect_ratio = max(h, w) / max(1, min(h, w))
    return dict(area=float(area), perimeter=float(perimeter),
                compactness=float(compactness), aspect_ratio=float(aspect_ratio))


def show_image(ax, image: np.ndarray, title: str = "", cmap: str = "gray"):
    """Show one image on a given Matplotlib axis, titled, with ticks removed."""
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


def show_image_grid(images, titles=None, ncols: int = 4, cmap: str = "gray", figsize_scale: float = 2.6):
    """A grid of images with Matplotlib, one subplot each, titles optional."""
    n = len(images)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_scale * ncols, figsize_scale * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, img in enumerate(images):
        show_image(axes[i], img, titles[i] if titles else "", cmap=cmap)
    for ax in axes[n:]:
        ax.axis("off")
    fig.tight_layout()
    return fig, axes

---

## 1 · Model zoos: a shared registry of interchangeable tools

Once there are many trained models for a given job (several generalist segmenters from Module 28,
say), picking the right one, with the right input/output conventions, becomes its own problem. The
**BioImage Model Zoo** (bioimage.io) solves this with a standard *specification*: every model in
the zoo ships with machine-readable metadata — what it expects as input, what it produces as
output, what it was trained on, how to cite it — so that any compatible tool (napari, ilastik,
Fiji, or an AI agent) can discover, check compatibility with, and run a model without a human
reading its paper first.

We build a miniature version of this idea: a small **tool registry** wrapping functions from this
course's own toolkit, each with the same kind of structured metadata a real model-zoo entry would
carry.

In [ ]:
# ── A miniature "model zoo": toolkit functions wrapped with structured metadata ──
TOOL_REGISTRY = {
    "simulate_image": {
        "fn": lambda **kw: simulate_nuclei_image(**kw),
        "inputs": {"n_cells": "int", "size": "int", "seed": "int"},
        "outputs": {"image": "float array (size, size)", "instance_mask": "int array",
                    "boxes": "array (n_cells, 4)"},
        "description": "Produce a SIMULATED fluorescence image of cell nuclei with known "
                        "ground-truth instance segmentation.",
    },
    "segment_watershed": {
        "fn": lambda image: watershed_labels(image),
        "inputs": {"image": "float array (H, W)"},
        "outputs": {"labels": "int array (H, W), 0 = background"},
        "description": "Classical marker-controlled watershed instance segmentation of a blobby "
                        "intensity image (Module 18).",
    },
    "measure_shape": {
        "fn": lambda mask: shape_features(mask),
        "inputs": {"mask": "boolean array (H, W)"},
        "outputs": {"area": "float", "perimeter": "float", "compactness": "float",
                    "aspect_ratio": "float"},
        "description": "Morphological shape features of a single binary object mask (Module 15).",
    },
}

for name, spec in TOOL_REGISTRY.items():
    print(f"• {name}: {spec['description']}")
    print(f"    inputs : {spec['inputs']}")
    print(f"    outputs: {spec['outputs']}")

🔍 **What just happened?** Nothing here is new *functionality* — every one of these
tools already existed in the course toolkit. What is new is the uniform wrapper: a name, a plain-
language description, and an explicit input/output contract. That uniformity is precisely what
lets something else — a viewer plugin, a workflow engine, or the agent in §2 — pick the right tool
for a job without a person reading source code first.

---

## 2 · An agent loop: plan, act, observe, verify

An AI agent for bioimage analysis takes a request in natural language ("segment the nuclei in this
stack and report their size distribution"), decides which tools from a registry like §1's to call
and in what order, calls them, and uses each result to decide the next step. This is usually called
a **ReAct**-style loop (Reason + Act): *Thought* (what should I do next and why), *Action* (call
one tool), *Observation* (what came back), repeated until the request is answered.

The "agent" below is a small, fully scripted Python program, not a live language model — this
notebook has no network access, and more importantly, a scripted agent lets every step of the loop
be inspected and checked exactly, which is the whole point of this course's "trust but verify"
theme from Module 34.

In [ ]:
# ── A tiny, fully scripted ReAct-style agent over the TOOL_REGISTRY ──
def run_bioimage_agent(request, seed=0, verbose=True):
    """A deterministic stand-in for an LLM-driven agent: a fixed plan for this one class of
    request ("segment + report size distribution"), executed step by step with an explicit
    Thought / Action / Observation trace, and a verification check before trusting each result."""
    trace = []

    def log(role, text):
        trace.append((role, text))
        if verbose:
            print(f"[{role}] {text}")

    log("Thought", f'Request: "{request}". This asks for (1) an image, (2) its segmentation, '
                    "(3) per-object size features, (4) a summary. Plan: simulate_image -> "
                    "segment_watershed -> measure_shape (per object) -> summarize.")

    log("Action", "simulate_image(n_cells=20, size=160, seed=seed)")
    sim = TOOL_REGISTRY["simulate_image"]["fn"](n_cells=20, size=160, seed=seed)
    log("Observation", f"got an image {sim['image'].shape} with {sim['instance_mask'].max()} "
                        "labeled objects (ground truth, since this is our own simulator)")

    log("Action", "segment_watershed(image)")
    labels = TOOL_REGISTRY["segment_watershed"]["fn"](sim["image"])
    n_found = labels.max()
    log("Observation", f"watershed found {n_found} objects")

    # ── Verify before trusting the next step (the Module 34 habit, applied to a pipeline) ──
    true_n = sim["instance_mask"].max()
    if abs(n_found - true_n) > max(2, 0.25 * true_n):
        log("Verify", f"⚠️ segmentation count ({n_found}) is far from the simulator's own object "
                       f"count ({true_n}) — flagging this result instead of trusting it blindly.")
        return dict(trace=trace, status="FLAGGED", reason="segmentation count mismatch")
    log("Verify", f"segmentation count ({n_found}) is close to plausible ({true_n} objects were "
                  "actually placed) — proceeding.")

    log("Action", f"measure_shape(mask) for each of the {n_found} segmented objects")
    areas = []
    for obj_id in range(1, n_found + 1):
        feats = TOOL_REGISTRY["measure_shape"]["fn"](labels == obj_id)
        areas.append(feats["area"])
    areas = np.array(areas)
    log("Observation", f"measured {len(areas)} objects; "
                        f"area range [{areas.min():.0f}, {areas.max():.0f}] px")

    answer = (f"Found {n_found} segmented objects. Size distribution: mean area "
              f"{areas.mean():.0f} px (SD {areas.std():.0f}), median {np.median(areas):.0f} px.")
    log("Answer", answer)
    return dict(trace=trace, status="OK", areas=areas, n_found=n_found, answer=answer)


result = run_bioimage_agent("segment the nuclei in this image and report their size distribution",
                             seed=SEED)

🧪 **Ground-truth check.** The agent's `Verify` step is not decorative: because
`simulate_image` is our own simulator, we know the true number of objects it placed
(`sim["instance_mask"].max()`), so the agent can check its own intermediate result against ground
truth *before* spending effort measuring shapes on a segmentation that might be badly wrong. A
real pipeline rarely has this luxury (no simulator to check against) — which is exactly why
Module 34's auditing habits (a held-out sanity set, a plausibility range, a human spot-check)
matter even more once an agent is chaining steps unattended.

In [ ]:
# ── 🧪 What happens when a step actually goes wrong? Force a bad segmentation ──
def broken_segment(image):
    """Deliberately useless 'segmentation': everything is one giant blob. Stands in for a real
    failure mode — e.g. a threshold tuned for the wrong imaging condition (Module 34)."""
    return (image > image.min()).astype(int)


def run_bioimage_agent_with_broken_tool(request, seed=0, verbose=True):
    trace = []

    def log(role, text):
        trace.append((role, text))
        if verbose:
            print(f"[{role}] {text}")

    log("Thought", "Same plan as before, but segment_watershed has been swapped for a broken tool.")
    sim = TOOL_REGISTRY["simulate_image"]["fn"](n_cells=20, size=160, seed=seed)
    log("Action", "segment_watershed(image)  [actually: broken_segment]")
    labels = broken_segment(sim["image"])
    n_found = labels.max()
    log("Observation", f"'watershed' found {n_found} object(s)")
    true_n = sim["instance_mask"].max()
    if abs(n_found - true_n) > max(2, 0.25 * true_n):
        log("Verify", f"⚠️ segmentation count ({n_found}) is far from plausible ({true_n} objects "
                       "were actually placed) — STOPPING instead of reporting a size distribution "
                       "computed from a single giant blob.")
        return dict(trace=trace, status="FLAGGED")
    log("Answer", "(never reached)")
    return dict(trace=trace, status="OK")


broken_result = run_bioimage_agent_with_broken_tool(
    "segment the nuclei in this image and report their size distribution", seed=SEED)
print(f"\nfinal status: {broken_result['status']}")

🔍 **What just happened?** Without the verification step, the broken agent would have
confidently reported a "size distribution" computed from exactly one giant blob — a wrong answer
delivered with no warning, the same silent-failure pattern Module 34 built an entire notebook
around. With the verification step, the agent catches the implausible intermediate result and
stops rather than compounding it into a confident wrong answer. This is the single most important
practical lesson in this module: **an agent's output is only as trustworthy as the checks placed
between its steps.**

✏️ **Your turn.** Loosen the verification tolerance (change `0.25 * true_n` to, say,
`2.0 * true_n`) and re-run the broken-tool agent. At what tolerance does the check stop catching
the obviously wrong single-blob segmentation? This is exactly the kind of threshold-choice
trade-off Module 34's reproducibility section warned about — too loose a check lets errors
through silently; too tight a check flags correct results as suspicious.

---

## 3 · Retrieval-augmented analysis assistants

A real agent is also frequently asked *which tool to use* for a stated need, not just to run a
fixed plan. Answering that from a language model's unaided memory risks a confident, plausible-
sounding, *wrong* recommendation — the bioimage-analysis version of hallucination. A
**retrieval-augmented** assistant instead searches real documentation text and grounds its answer
in what it actually finds, the same idea behind retrieval-augmented generation (RAG) systems more
broadly.

In [ ]:
# ── A tiny retrieval step grounded in real toolkit documentation strings ──
KNOWLEDGE_BASE = {
    "watershed_labels": "Marker-controlled watershed segmentation of a blobby intensity image: "
                         "finds local-maximum markers and floods outward from them. Use this for "
                         "splitting touching round objects like nuclei.",
    "hungarian_assignment": "The Hungarian algorithm for the linear assignment problem: the "
                             "lowest-total-cost one-to-one matching between two sets of points. "
                             "Use this for frame-to-frame cell tracking.",
    "pearson_colocalization": "Pearson's colocalization coefficient: ordinary linear correlation "
                               "between two fluorescence channels' pixel intensities. Use this to "
                               "ask whether two proteins' signals rise and fall together.",
    "ripley_l": "Ripley's L function: compares the observed density of neighbors around each "
                "point, at a range of radii, to what complete spatial randomness would predict. "
                "Use this to ask whether cells are spatially clustered or dispersed.",
    "wiener_deconvolve": "Wiener deconvolution: inverse-filters a blurred, noisy image given its "
                         "point-spread function. Use this to sharpen an out-of-focus or "
                         "diffraction-blurred image when the blur kernel is known.",
}


def retrieve(query, kb=KNOWLEDGE_BASE):
    """A minimal keyword-overlap retriever: score every KB entry by how many of the query's
    (lower-cased) words appear in its text, and return the best match."""
    query_words = set(query.lower().replace("?", "").split())
    scores = {name: len(query_words & set(text.lower().replace(".", "").replace(",", "").split()))
              for name, text in kb.items()}
    best = max(scores, key=scores.get)
    return best, scores[best]


TEST_QUERIES = [
    ("which tool splits two touching round nuclei in one image?", "watershed_labels"),
    ("how do I match cells between this frame and the next frame?", "hungarian_assignment"),
    ("are these two fluorescence channels correlated with each other?", "pearson_colocalization"),
    ("is my point pattern more clustered than random chance would predict?", "ripley_l"),
    ("my image is blurry from a known point-spread function, how do I sharpen it?",
     "wiener_deconvolve"),
]

correct = 0
for query, expected in TEST_QUERIES:
    retrieved, score = retrieve(query)
    is_correct = retrieved == expected
    correct += is_correct
    print(f"{'✓' if is_correct else '✗'} query: {query!r}\n    retrieved: {retrieved} "
          f"(score {score})  expected: {expected}")
print(f"\nretrieval accuracy on {len(TEST_QUERIES)} fixed test queries: "
      f"{correct}/{len(TEST_QUERIES)}")

🧪 **Ground-truth check.** We wrote `TEST_QUERIES` with a known correct answer for
each one, so "did retrieval get it right" is not a matter of opinion — it is exactly the kind of
offline evaluation set a real retrieval-augmented system is checked against before deployment. A
keyword-overlap retriever is the simplest possible version of this idea; production RAG systems
use dense vector embeddings and semantic similarity instead of literal word overlap, precisely
because keyword overlap fails on queries that mean the same thing in different words (try it:
what happens to `retrieve("tell me whether two proteins sit in the same place")` — note it shares
almost no literal words with the `pearson_colocalization` entry)?

---

## 4 · Fusing imaging with multi-omics and clinical metadata

Module 31 introduced joint reasoning over an image and a second data modality (transcriptomic or
proteomic measurements at the same spatial location). The same idea extends to clinical metadata —
age, treatment, outcome — paired with per-cell or per-patient imaging features. The claim worth
testing concretely: sometimes **neither modality alone carries the answer, but the two together
do.** We build the sharpest possible toy version of this claim — not because real biology is
usually this extreme, but because it makes the point unmistakable and fully checkable.

In [ ]:
# ── 🧪 A case where NEITHER modality alone predicts the label, but both together can ──
N = 400
hidden_subtype = rng.choice([-1.0, 1.0], size=N)       # an unobserved tumor subtype
hidden_pathway = rng.choice([-1.0, 1.0], size=N)       # an unobserved expression-pathway state

# An image-derived feature (e.g. nuclear shape irregularity) that reflects the hidden subtype...
image_feature = hidden_subtype + rng.normal(0, 0.6, size=N)
# ...and an omics-derived feature (e.g. a pathway activity score) that reflects the hidden pathway.
omics_feature = hidden_pathway + rng.normal(0, 0.6, size=N)

# The clinical label is TRUE exactly when the subtype and pathway state DISAGREE (an XOR) — a
# deliberately extreme case where the label has ZERO marginal correlation with either feature
# alone, but is fully determined by the combination of both.
label = (hidden_subtype * hidden_pathway < 0).astype(np.float32)

split = int(N * 0.75)
Xi_tr, Xi_te = image_feature[:split], image_feature[split:]
Xo_tr, Xo_te = omics_feature[:split], omics_feature[split:]
y_tr, y_te = label[:split], label[split:]

print(f"correlation of image_feature alone with label : {np.corrcoef(image_feature, label)[0,1]:+.3f}")
print(f"correlation of omics_feature alone with label  : {np.corrcoef(omics_feature, label)[0,1]:+.3f}")

In [ ]:
# ── Single-modality linear classifiers vs. a joint non-linear classifier ──
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(SEED)


def train_logistic(X, y, n_epochs=300, lr=0.1):
    model = nn.Linear(X.shape[1], 1)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    Xt = torch.from_numpy(X).float()
    yt = torch.from_numpy(y).float().unsqueeze(1)
    for _ in range(n_epochs):
        opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(model(Xt), yt)
        loss.backward()
        opt.step()
    return model


def train_joint_mlp(X, y, n_epochs=400, lr=0.05, hidden=8):
    model = nn.Sequential(nn.Linear(X.shape[1], hidden), nn.Tanh(), nn.Linear(hidden, 1))
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    Xt = torch.from_numpy(X).float()
    yt = torch.from_numpy(y).float().unsqueeze(1)
    for _ in range(n_epochs):
        opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(model(Xt), yt)
        loss.backward()
        opt.step()
    return model


def test_accuracy(model, X, y):
    with torch.no_grad():
        pred = (torch.sigmoid(model(torch.from_numpy(X).float())) > 0.5).numpy().ravel()
    return (pred == y).mean()


model_image_only = train_logistic(Xi_tr[:, None], y_tr)
model_omics_only = train_logistic(Xo_tr[:, None], y_tr)
X_joint_tr = np.stack([Xi_tr, Xo_tr], axis=1)
X_joint_te = np.stack([Xi_te, Xo_te], axis=1)
model_joint = train_joint_mlp(X_joint_tr, y_tr)

acc_image = test_accuracy(model_image_only, Xi_te[:, None], y_te)
acc_omics = test_accuracy(model_omics_only, Xo_te[:, None], y_te)
acc_joint = test_accuracy(model_joint, X_joint_te, y_te)

print(f"image feature alone  (linear) : {acc_image:.1%} held-out accuracy")
print(f"omics feature alone  (linear) : {acc_omics:.1%} held-out accuracy")
print(f"BOTH features jointly (small non-linear model): {acc_joint:.1%} held-out accuracy")
print("(chance = 50%)")

🧪 **Ground-truth check.** We built `label` ourselves as an exact function of the two
hidden variables, so "does joint reasoning actually help" is a number, not an impression: the two
single-modality linear models should land close to the 50% chance rate stated above (by
construction, each feature alone has zero marginal correlation with the label), while the joint
model — which sees both features *and* has enough capacity (a hidden layer) to represent their
interaction — should score well above chance. Note the second half of that claim is important: a
**linear** model given both raw features still cannot solve this particular toy problem (the
classic reason XOR is used to teach why linear models have limited capacity) — joint data is
necessary but, here, not sufficient without a model able to represent the interaction between the
modalities, not just add them up.

This toy example is deliberately extreme (a real image-plus-omics relationship is
rarely a pure XOR). What it demonstrates cleanly is the general shape of the real argument from
Module 31's spatial-omics integration: a joint analysis can expose structure that is invisible —
not just somewhat harder to see, but mathematically absent as a marginal signal — in either data
modality examined alone.

---

## 5 · Responsible use: verification, dual-use and privacy

Everything above compounds a real concern from Module 34: an agent that chains several models
together multiplies the number of places a silent failure can enter, and multiplies how far a
wrong intermediate result can travel before a human notices. Three practical habits scale with
pipeline complexity rather than shrink:

- **Verify every tool's output before the next step trusts it** — §2's segmentation-count check is
  the simplest possible version of this; a real pipeline needs one such check per step, ideally
  tied to a quantity you can actually sanity-check (a plausible count, a plausible range, a
  held-out ground-truth comparison where one exists).
- **Never treat an agent's final answer as verified just because it sounds coherent** — a fluent,
  confident summary is not evidence the underlying numbers were ever right; always spot-check
  against an independent source (a manual count on a few images, a known control).
- **Dual-use and privacy do not go away because a human did not personally run every step** — an
  automated pipeline that reasons over imaging plus clinical metadata (§4) is still handling
  potentially identifying or sensitive data, and an agent empowered to act (file a report, flag a
  patient, trigger a downstream decision) needs the same oversight and consent safeguards a human
  analyst doing the same job would need, arguably more, precisely because it can act faster and at
  greater scale than a person reviewing each case.

---

## 6 · 🧭 From toy to real data

### 📄 Research connection — the BioImage Model Zoo
Ouyang, W. et al. (2022). *BioImage.IO: a community-driven, discoverable model zoo for sharing
model weights, specifications, and automated analysis for bioimaging.* Nature Methods (the
specific real-world model zoo §1's toy `TOOL_REGISTRY` is modeled after).

### 📄 Research connection — ReAct-style agent loops
Yao, S. et al. (2023). *ReAct: Synergizing Reasoning and Acting in Language Models.* ICLR. The
Thought/Action/Observation loop implemented by hand (without a real language model) in §2.

### 📄 Research connection — AI agents for scientific experimentation
Boiko, D. A. et al. (2023). *Autonomous chemical research with large language models.* Nature;
Bran, A. M. et al. (2024). *Augmenting large language models with chemistry tools.* Nature
Machine Intelligence. Two concrete, published examples of agents orchestrating real laboratory and
computational tools — and, notably, both papers discuss verification and safety guardrails
extensively, for exactly the reasons §2 and §5 raised here.

### 📄 Research connection — retrieval-augmented generation
Lewis, P. et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.*
NeurIPS. The general framework behind §3's toy retriever — grounding a generated answer in
retrieved text rather than a language model's unaided (and sometimes wrong) memory.

### 📄 Research connection — joint imaging and multi-omics integration
Moses, L. & Pachter, L. (2022). *Museum of spatial transcriptomics.* Nature Methods (a survey of
the broader spatial multi-omics landscape §4's toy example is a deliberately simplified stand-in
for) — see also Module 31's own references for spatial-omics-specific integration methods.

At production scale, §2's hand-scripted agent becomes a real language model with function-calling
or tool-use support, reasoning over a model zoo like bioimage.io's real catalog; §3's keyword
retriever becomes a dense-embedding vector search over real documentation and papers; and §4's toy
XOR becomes a multimodal model (often itself a small neural network taking imaging and omics
embeddings as joint input) trained on real paired cohort data, with all of Module 34's auditing
practices applied to every stage.

## 📌 Key takeaways

- A model zoo's value is almost entirely in its standardized metadata, not in any single model —
  it is what lets a viewer, a workflow engine, or an agent discover and use the right tool without
  a human reading documentation first.
- An agentic plan-act-observe loop is the same shape whether it is a hand-scripted program (this
  notebook) or a language model with tool access — and either way, a verification check between
  steps is what keeps one bad intermediate result from silently becoming a confidently wrong final
  answer.
- Retrieval grounds an answer in text you can actually check, rather than a model's unaided
  memory, and its quality is itself measurable against a fixed set of test queries with known
  correct answers.
- Some questions genuinely cannot be answered from either data modality alone — joint reasoning
  can expose structure invisible to any single modality, provided the model has enough capacity to
  represent how the modalities interact, not just concatenate them.
- Complexity multiplies the number of places a pipeline can silently fail, which makes
  Module 34's verification habits more important, not optional, once an agent is doing the
  chaining.

## ✅ Checkpoint

**1. In §2, why does the agent check the segmentation count against the simulator's own ground
truth rather than, say, just checking that `segment_watershed` ran without raising an exception?**

<details><summary>Answer</summary>

A tool can run successfully and still return a badly wrong result — the broken-tool example in
§2 produces exactly one "object" (the whole image as one blob) with no error raised at all. Only
checking for an exception would let this pass silently; checking the *result* against a
plausibility range (or, where available, ground truth) is what actually catches a wrong-but-
successful tool call, which is the far more common and far more dangerous failure mode in
practice.
</details>

**2. Why does a keyword-overlap retriever fail on a paraphrased query that uses none of the same
words as the correct knowledge-base entry, and what would fix it?**

<details><summary>Answer</summary>

Keyword overlap can only ever match literal shared words, so a query like "tell me whether two
proteins sit in the same place" shares almost no words with an entry about "Pearson's
colocalization coefficient... two fluorescence channels," even though they mean the same thing.
The fix is semantic retrieval: embed both the query and the knowledge-base entries into a vector
space (using a trained text-embedding model) where paraphrases land close together, and retrieve
by vector similarity instead of literal word overlap — the standard approach in production
retrieval-augmented systems.
</details>

**3. In §4, why does a *linear* model fail even when it is given both features, and what does that
imply about choosing between "collect more modalities" and "use a more expressive model"?**

<details><summary>Answer</summary>

The XOR-style label is not linearly separable in the two raw features' space — no straight line
(or hyperplane, in higher dimensions) can separate the two classes, regardless of how much data a
linear model sees. This implies the two concerns are genuinely separate: having the right data
(both modalities) is necessary but not sufficient; the model fitted to that data also needs enough
representational capacity (here, a hidden layer) to capture how the modalities interact. Neither
fix substitutes for the other.
</details>

## 🧠 Questions & discussion

- The agent in §2 checks one intermediate result against a known ground truth that only exists
  because this is a simulated course exercise. In a real pipeline with no simulator to check
  against, what would you use in place of ground truth — and how confident could you ever be in
  that substitute?
- If an agent is given the ability not just to analyze images but to *act* on the results (flag a
  sample for review, trigger a re-scan, notify a clinician), at what point in the pipeline would
  you insist on a human reviewing the agent's reasoning before that action is taken, and why there
  specifically?
- Multimodal fusion (§4) often requires linking a patient's imaging data to their omics and
  clinical records — data that was very likely collected, consented to, and governed separately,
  by different parts of an institution. What practical and ethical obstacles does linking these
  datasets raise that do not exist when analyzing imaging data alone?

## 🚀 Bonus — going further

- Extend `TOOL_REGISTRY` with a fourth tool (e.g. `track_cells` wrapping `hungarian_assignment`)
  and extend `run_bioimage_agent` to handle a new request type ("track the cells across these
  frames and report how many divided") that chains a different sequence of tools.
- Replace the keyword-overlap `retrieve` function with an embedding-based one (e.g. using a small
  pretrained sentence-embedding model, if you have one available) and re-run `TEST_QUERIES` —
  does it now also get the paraphrased query from the Checkpoint right?
- Make the multimodal toy example less extreme: instead of a pure XOR, make the label a *weighted
  sum* of the two hidden variables plus noise (a case where each modality alone carries partial,
  genuinely additive information). Does a simple linear model now succeed on both single
  modalities, just less well than the joint model — unlike the all-or-nothing XOR case?

## 📝 Lab note

Record, in your own words:

1. One sentence: what specifically does the verification step in §2 protect against that a plain
   "did the function raise an error" check would miss?
2. One number: the retrieval accuracy you measured in §3, and one example query you predict would
   break the keyword-overlap retriever (not already in `TEST_QUERIES`).
3. One claim backed by a number: the held-out accuracy gap between the best single-modality model
   and the joint model in §4, and one sentence on why that gap is a worst-case (not typical-case)
   illustration of the real argument.

## 📚 References

1. Ouyang, W. et al. (2022). BioImage.IO: a community-driven, discoverable model zoo for sharing
   model weights, specifications, and automated analysis for bioimaging. *Nature Methods*.
2. Yao, S. et al. (2023). ReAct: Synergizing Reasoning and Acting in Language Models. *ICLR*.
3. Boiko, D. A. et al. (2023). Autonomous chemical research with large language models. *Nature*.
4. Bran, A. M. et al. (2024). Augmenting large language models with chemistry tools. *Nature
   Machine Intelligence*.
5. Lewis, P. et al. (2020). Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.
   *NeurIPS*.
6. Moses, L. & Pachter, L. (2022). Museum of spatial transcriptomics. *Nature Methods*.
7. Minsky, M. & Papert, S. (1969). *Perceptrons*. MIT Press (the classical source of the XOR
   linear-separability argument used in §4).

## ➡️ Next

Module 36 is the capstone: one complete pipeline, built from scratch by chaining this course's own
toolkit end to end, with a project studio of extension tracks and a reproducibility checklist
drawing on everything from Modules 32 through 35.